# Tool-calling lab: a safe first lesson

This notebook is a small, offline fixture. It teaches the data contract, strict parser, evaluation metrics, masking idea, and LoRA arithmetic. It does not download a dataset or model, train anything, execute a tool, start an agent loop, or initialize W&B.

Run the cells from the repository root. Every example stays in memory. Real pipeline commands belong in [the tutorial](tutorial.md).

In [ ]:
from pathlib import Path
import json

from ftlab.data import AcceptedRecord, normalize_record
from ftlab.metrics import evaluate_predictions, no_call_accuracy
from ftlab.parser import parse_prediction
from ftlab.rendering import render_json_call, render_record

weather_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "units": {"type": "string", "enum": ["celsius", "fahrenheit"]},
    },
    "required": ["city"],
    "additionalProperties": False,
}
weather_fixture = {
    "id": "lesson-weather-1",
    "query": "What is the weather in Paris in Celsius?",
    "tools": [
        {
            "name": "weather.get",
            "description": "Return current weather for a city.",
            "parameters": weather_schema,
        }
    ],
    "answers": [{"name": "weather.get", "arguments": {"city": "Paris", "units": "celsius"}}],
}
weather_record = normalize_record(weather_fixture, "lesson-weather-1")
assert isinstance(weather_record, AcceptedRecord)
normalized_schema = weather_record.tools[0]["parameters"]
gold_call = {"name": weather_record.function_name, "arguments": weather_record.arguments}
print("Normalized tool:", weather_record.tools[0]["name"])
print("Normalized gold call:", gold_call)

## Render the prompt and completion

A prompt is the text sent to the language model. A completion is the text it generates. The fallback renderer below uses ordinary text, not a tokenizer. It makes the contract visible:

```text
<tool_call>
{"name": "weather.get", "arguments": {"city": "Paris", "units": "celsius"}}
</tool_call>
```

The host application would parse this text. The notebook never calls `weather.get`.

In [ ]:
rendered = render_record(weather_record)
prompt_text = rendered.prompt
completion_text = rendered.completion
full_text = rendered.full
assert full_text == prompt_text + completion_text
assert completion_text.startswith("<tool_call>") and completion_text.endswith("</tool_call>")
print("Prompt characters:", len(prompt_text))
print("Completion:", completion_text)

## Prompt masking with a teaching tokenizer

During fine-tuning, prompt positions can be masked with `-100` so their loss is ignored. Only completion targets teach the model. This cell uses characters as teaching tokens. Real Qwen tokenization uses learned subword tokens and special tokens, so these positions are an explanation, not a replacement for the real tokenizer.

In [ ]:
teaching_prompt_tokens = list(prompt_text)
teaching_completion_tokens = list(completion_text)
teaching_tokens = teaching_prompt_tokens + teaching_completion_tokens
masked_labels = [-100] * len(teaching_prompt_tokens) + teaching_completion_tokens
assert masked_labels[: len(teaching_prompt_tokens)] == [-100] * len(teaching_prompt_tokens)
assert masked_labels[len(teaching_prompt_tokens) :] == teaching_completion_tokens
masking_summary = {
    "teaching_tokenizer": "one character per teaching token",
    "prompt_positions": len(teaching_prompt_tokens),
    "completion_targets": len(teaching_completion_tokens),
    "masked_label": -100,
}
print("Masking summary:", masking_summary)

## Strict parsing

The parser accepts exactly one complete `<tool_call> ... </tool_call>` boundary. It rejects thinking blocks, incomplete or multiple boundaries, extra text, invalid JSON (including duplicate keys and non-finite numbers), non-object payloads or arguments, extra call fields, and missing or unknown tools. Schema validation is a separate metric diagnostic after JSON and tool parsing.

In [ ]:
correct_call = completion_text
parser_examples = {
    "thinking_block": "<think>reasoning</think>" + correct_call,
    "partial_boundary": correct_call[: -len("</tool_call>")],
    "extra_text": "prefix " + correct_call,
    "invalid_json": "<tool_call>{\"name\":\"weather.get\",\"arguments\":{\"city\":\"Paris\",\"city\":\"Delhi\"}}</tool_call>",
    "nonobject_payload": "<tool_call>[]</tool_call>",
    "nonobject_arguments": "<tool_call>{\"name\":\"weather.get\",\"arguments\":[]}</tool_call>",
    "extra_call_field": "<tool_call>{\"name\":\"weather.get\",\"arguments\":{},\"extra\":true}</tool_call>",
    "unknown_tool": "<tool_call>{\"name\":\"weather.other\",\"arguments\":{\"city\":\"Paris\"}}</tool_call>",
    "schema_failure": "<tool_call>{\"name\":\"weather.get\",\"arguments\":{\"city\":42}}</tool_call>",
    "multiple_calls": correct_call + correct_call,
}
parsed_correct = parse_prediction(correct_call, weather_record.tools)
parsed_examples = {label: parse_prediction(text, weather_record.tools) for label, text in parser_examples.items()}
parse_categories = {label: parsed.error.category for label, parsed in parsed_examples.items() if parsed.error}
expected_categories = {
    "thinking_block": "thinking_block",
    "partial_boundary": "partial_boundary",
    "extra_text": "extra_output",
    "invalid_json": "invalid_json",
    "nonobject_payload": "json_fragment",
    "nonobject_arguments": "non_object_arguments",
    "extra_call_field": "invalid_call",
    "unknown_tool": "unknown_tool",
    "schema_failure": "schema_invalid",
    "multiple_calls": "multiple_calls",
}
assert parsed_correct.valid
assert parse_categories == expected_categories
print("Parser categories:", parse_categories)

## Metrics on a small evaluation batch

This batch has four examples: one exact call, one partial but schema-valid call, one schema-valid call with wrong values, and one parser failure. JSON, tool, schema, and correctness metrics remain independent; for example, schema-invalid JSON can still be JSON-valid and name a supplied tool. A parser failure has zero validity and correctness for that record. `task_success` requires the correct tool, schema-valid arguments, and correct required gold arguments. It is separate from exact-object matching.

Argument key and value F1 use nested leaf paths and aggregate their counts across this batch. Strings are trimmed and normalized to Unicode NFC. Integers and floats compare by value only when the schema says `number`; booleans never compare as numbers.

In [ ]:
metric_predictions = [
    correct_call,
    render_json_call("weather.get", {"city": "Paris"}),
    render_json_call("weather.get", {"city": "London", "units": "fahrenheit"}),
    "<think>not allowed</think>",
]
evaluation_records = [weather_record] * len(metric_predictions)
evaluation = evaluate_predictions(evaluation_records, metric_predictions)
metric_values = evaluation.as_dict()
no_call_score = no_call_accuracy(["A normal answer", correct_call])
assert evaluation.count == 4
assert evaluation.tool_accuracy == 0.75
assert evaluation.json_validity == 0.75
assert evaluation.schema_validity == 0.75
assert evaluation.exact_match == 0.25
assert evaluation.task_success == 0.5
assert evaluation.parser_errors == {"thinking_block": 1}
assert no_call_score == 0.5
print("Evaluation:", metric_values)
print("No-call accuracy (robustness only):", no_call_score)

## LoRA arithmetic and optimizer updates

An adapter changes a frozen weight matrix with `W' = W + scale × B A`. MLX-LM applies the configured `scale` directly; this project does not describe it as an `alpha/r` conversion. For a rank-`r` adapter on a matrix with input width `in` and output width `out`, the trainable count is `r × (in + out)`.

The pinned architecture evidence is hidden size 1024, 16 attention heads, 8 key/value heads, and head dimension 128. Therefore q output is 2048 and v output is 1024. The final eight layers adapt q and v only.

In [ ]:
hidden_size = 1024
attention_heads = 16
key_value_heads = 8
head_dim = 128
lora_rank = 8
num_layers = 8
q_output = attention_heads * head_dim
v_output = key_value_heads * head_dim
q_lora_parameters = lora_rank * (hidden_size + q_output)
v_lora_parameters = lora_rank * (hidden_size + v_output)
per_layer_lora_parameters = q_lora_parameters + v_lora_parameters
lora_module_count = num_layers * 2
total_lora_parameters = num_layers * per_layer_lora_parameters
assert (q_output, v_output) == (2048, 1024)
assert (q_lora_parameters, v_lora_parameters) == (24576, 16384)
assert per_layer_lora_parameters == 40960
assert lora_module_count == 16
assert total_lora_parameters == 327680
lora_counts = {
    "q_output": q_output,
    "v_output": v_output,
    "q_per_layer": q_lora_parameters,
    "v_per_layer": v_lora_parameters,
    "modules": lora_module_count,
    "total": total_lora_parameters,
}
print("LoRA counts:", lora_counts)

In [ ]:
microbatches = 128
batch_size = 1
gradient_accumulation_steps = 8
optimizer_updates = microbatches // gradient_accumulation_steps
assert optimizer_updates == 16
update_summary = {
    "microbatches": microbatches,
    "batch_size": batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "optimizer_updates": optimizer_updates,
}
print("Update arithmetic:", update_summary)

## Inspect the checked-in experiment configuration

This reads only the small YAML configuration already in the repository. It does not fetch or initialize anything.

In [ ]:
import yaml

config_path = Path("configs/experiments/smoke-06b.yaml")
experiment_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
assert experiment_config["model"] == "Qwen/Qwen3-0.6B-MLX-4bit"
assert experiment_config["model_revision"] == "173234aa840d113125e9f2271100ddbaf16c9620"
assert experiment_config["dataset"]["revision"] == "26d14ebfe18b1f7b524bd39b404b50af5dc97866"
assert experiment_config["num_layers"] == 8
assert experiment_config["grad_accumulation_steps"] == 8
config_values = {
    "model": experiment_config["model"],
    "model_revision": experiment_config["model_revision"],
    "dataset_revision": experiment_config["dataset"]["revision"],
    "max_seq_length": experiment_config["max_seq_length"],
    "lora_scale": experiment_config["lora_parameters"]["scale"],
}
print("Configuration values:", config_values)

In [ ]:
manifest_path = Path("data/smoke/manifest.json")
manifest_summary = {"exists": manifest_path.exists()}
if manifest_path.exists():
    manifest_value = json.loads(manifest_path.read_text(encoding="utf-8"))
    split_value = manifest_value.get("splits", {})
    split_counts = {
        name: len(rows) for name, rows in split_value.items() if isinstance(rows, list)
    }
    flow_value = manifest_value.get("flow", {})
    flow_counts = {
        name: flow_value[name]
        for name in ("source", "accepted", "rejected")
        if isinstance(flow_value, dict) and isinstance(flow_value.get(name), int)
    }
    manifest_summary.update(
        {"split_names": sorted(split_counts), "split_counts": split_counts, "flow_counts": flow_counts}
    )
    print("Optional manifest structure:", manifest_summary)
else:
    manifest_summary["message"] = "No optional smoke manifest is present; this is safe."
    print(manifest_summary["message"])

## What to do next

Use the tutorial for the real, pinned pipeline. Its command cells are intentionally separate from this fixture notebook. Try changing the weather answer, adding a parser rejection, or changing one metric prediction and predict the metric changes before running the cell.